# Noisy Minimum-Norm Path-Form Inversion — Demo Notebook

This notebook builds toward a **noisy minimum-norm inversion** workflow in which measurements are **path integrals over a 3D Sobolev model space**.

We use:
- A **unit-cube domain** $\Omega = [0,1]^3$
- A **low-order `Sobolev3D` model space** assembled from `BoxLaplacianSpectrumProvider` and `BesselSobolevBox`
- **10 random straight-ray paths** with sources drawn from every boundary face *except* the top face ($z = z_{\max}$) and receivers placed on the top face

**Phase 1 (this notebook):** Domain, model space, and acquisition geometry.  
**Phase 2** (future): Path-integral forward operator (`PathIntegralFunctional`, `SOLAOperator3D`) and noisy synthetic data.  
**Phase 3** (future): Noisy minimum-norm inversion via `LinearMinimumNormInversion` with a data covariance model.  
**Phase 4** (future): Model comparison visualizations and concluding discussion.

> **Note:** `PathIntegralFunctional` requires a `Sobolev3D` domain (not a plain `Lebesgue3D`) due to the unboundedness of 1-D curve evaluations in $L^2(\Omega)$. This constraint is enforced by the API and is exploited in Phase 2.

In [6]:
import numpy as np
import matplotlib.pyplot as plt

from pygeoinf import EuclideanSpace

from pygeoinf3d import (
    BoxDomain,
    Lebesgue3D,
    Path3D,
    BoxLaplacianSpectrumProvider,
    BesselSobolevBox,
    Sobolev3D,
)

# ── Reproducibility ────────────────────────────────────────────────────────
np.random.seed(42)
rng = np.random.default_rng(42)

---
## 1. Unit-Cube Domain and Low-Order Sobolev3D Model Space

We build:
1. A **unit-cube `BoxDomain`** — the physical domain $\Omega = [0,1]^3$.
2. A **`Lebesgue3D` base space** — the $L^2(\Omega)$ Hilbert space.
3. A **`BoxLaplacianSpectrumProvider`** with a coarse Neumann truncation — provides the spectral skeleton for the Sobolev weight.
4. A **low-order `Sobolev3D`** space assembled from `BesselSobolevBox` operators $W = (\tau - \Delta)^{s/2}$ with order $s = 1.75$.

A Sobolev space of order $s > d/2 = 1.5$ ensures that point evaluation, and therefore path-integral forms, is bounded in 3D. We choose a deliberately low but admissible order $s = 1.75$ so later inversion remains smooth without becoming overly expensive.

In [7]:
# ── 1. Domain ─────────────────────────────────────────────────────────────
box        = BoxDomain(np.zeros(3), np.ones(3))
base_space = Lebesgue3D(box)

# ── 2. Spectral provider (coarse truncation — keeps Phase 1 fast) ──────────
TRUNCATION = (8, 8, 8)   # 8^3 = 512 Neumann modes
SHIFT      = 1.0         # ensures invertibility at the zero Laplacian mode
ORDER      = 1.75        # choose s > d/2 = 1.5 for bounded path evaluations

provider = BoxLaplacianSpectrumProvider(box, truncation=TRUNCATION, bc_type="neumann")

# ── 3. Sobolev3D model space ───────────────────────────────────────────────
W = BesselSobolevBox(base_space, provider, shift=SHIFT, order=ORDER)
W_inv = BesselSobolevBox(base_space, provider, shift=SHIFT, order=-ORDER)
sobolev = Sobolev3D(base_space, W, W_inv)

print(f"Domain:                {box!r}")
print(f"Spectral truncation:   {TRUNCATION}  →  {provider.dim} modes")
print(f"Sobolev order:         s = {ORDER}")
print(f"Weight shift:          τ = {SHIFT}")
print(f"Base space type:       {type(base_space).__name__}")
print(f"Model space type:      {type(sobolev).__name__}")

Domain:                BoxDomain(low=[0, 0, 0], high=[1, 1, 1])
Spectral truncation:   (8, 8, 8)  →  512 modes
Sobolev order:         s = 1.75
Weight shift:          τ = 1.0
Base space type:       Lebesgue3D
Model space type:      Sobolev3D


---
## 2. Acquisition Geometry

We model a surface-to-depth acquisition survey:

- **Sources** are placed on the **five non-top boundary faces** ($x_{\min}$, $x_{\max}$, $y_{\min}$, $y_{\max}$, $z_{\min}$) — representing side-wall and bottom illumination.
- **Receivers** are placed on the **top face** ($z = z_{\max} = 1$) — representing surface sensors.

Each source–receiver pair defines a **straight ray** (zero-offset `Path3D.from_polyline` with two waypoints).

### Face sampling

For a box with side lengths $L_x = L_y = L_z = 1$ all non-top faces have equal area $1 \times 1 = 1$, so sources are drawn with uniform probability across the five faces.  In general one would weight by face area.

In [8]:
N_PATHS = 10
lo, hi = box.bounding_box().low, box.bounding_box().high
Lx, Ly, Lz = hi - lo

# ── Source sampling: all faces except z_max ────────────────────────────────
_face_labels = ["x_min", "x_max", "y_min", "y_max", "z_min"]
_face_areas = np.array([
    Ly * Lz,   # x_min
    Ly * Lz,   # x_max
    Lx * Lz,   # y_min
    Lx * Lz,   # y_max
    Lx * Ly,   # z_min
])
_face_probs = _face_areas / _face_areas.sum()

_face_indices = rng.choice(len(_face_labels), size=N_PATHS, p=_face_probs)
source_face_names = [_face_labels[i] for i in _face_indices]

sources = np.empty((N_PATHS, 3))
for k, fi in enumerate(_face_indices):
    face = _face_labels[fi]
    if face == "x_min":
        sources[k] = [lo[0], rng.uniform(lo[1], hi[1]), rng.uniform(lo[2], hi[2])]
    elif face == "x_max":
        sources[k] = [hi[0], rng.uniform(lo[1], hi[1]), rng.uniform(lo[2], hi[2])]
    elif face == "y_min":
        sources[k] = [rng.uniform(lo[0], hi[0]), lo[1], rng.uniform(lo[2], hi[2])]
    elif face == "y_max":
        sources[k] = [rng.uniform(lo[0], hi[0]), hi[1], rng.uniform(lo[2], hi[2])]
    else:  # z_min
        sources[k] = [rng.uniform(lo[0], hi[0]), rng.uniform(lo[1], hi[1]), lo[2]]

# ── Receiver sampling: top face (z = z_max) ───────────────────────────────
receivers = np.column_stack([
    rng.uniform(lo[0], hi[0], size=N_PATHS),
    rng.uniform(lo[1], hi[1], size=N_PATHS),
    np.full(N_PATHS, hi[2]),
])

# ── Straight-ray paths ────────────────────────────────────────────────────
paths = [
    Path3D.from_polyline(np.array([sources[i], receivers[i]]), domain=box)
    for i in range(N_PATHS)
]

---
## 3. Geometry Summary

The table below gives per-path diagnostics: source face, arc length, and endpoint coordinates.  We verify that:
- all source $z$-coordinates satisfy $z < z_{\max}$ (sources not on the top face), and
- all receiver $z$-coordinates equal $z_{\max}$.

In [9]:
print(f"Box domain:            {box!r}")
print(f"Spectral modes:        {provider.dim}  (truncation {TRUNCATION})")
print(f"Sobolev order:         s = {ORDER},  shift τ = {SHIFT}")
print(f"Number of paths:       {N_PATHS}")
print()
print(f"{'Path':>5}  {'Source face':>10}  {'Arc length':>10}  "
      f"{'Source (x,y,z)':>24}  {'Receiver (x,y,z)':>26}")
print("-" * 85)
for i, (p, face) in enumerate(zip(paths, source_face_names)):
    sx, sy, sz = sources[i]
    rx, ry, rz = receivers[i]
    print(
        f"  {i+1:>3d}  {face:>10s}  {p.arclength:>10.4f}  "
        f"  ({sx:.3f}, {sy:.3f}, {sz:.3f})    ({rx:.3f}, {ry:.3f}, {rz:.3f})"
    )

# Sanity checks
assert all(sources[i, 2] < hi[2] for i in range(N_PATHS)), \
    "At least one source is on the top face — check sampling!"
assert all(abs(receivers[i, 2] - hi[2]) < 1e-12 for i in range(N_PATHS)), \
    "At least one receiver is not on z_max!"

print()
print("✓  All sources have z < z_max  (not on the top face).")
print("✓  All receivers lie on z = z_max.")

Box domain:            BoxDomain(low=[0, 0, 0], high=[1, 1, 1])
Spectral modes:        512  (truncation (8, 8, 8))
Sobolev order:         s = 1.75,  shift τ = 1.0
Number of paths:       10

 Path  Source face  Arc length            Source (x,y,z)            Receiver (x,y,z)
-------------------------------------------------------------------------------------
    1       y_max      0.6797    (0.371, 1.000, 0.927)    (0.745, 0.437, 1.000)
    2       y_min      0.9108    (0.644, 0.000, 0.823)    (0.968, 0.833, 1.000)
    3       z_min      1.1125    (0.443, 0.227, 0.000)    (0.326, 0.700, 1.000)
    4       y_max      1.1761    (0.555, 1.000, 0.064)    (0.370, 0.312, 1.000)
    5       x_min      0.5968    (0.000, 0.828, 0.632)    (0.470, 0.832, 1.000)
    6       z_min      1.2353    (0.758, 0.355, 0.000)    (0.189, 0.805, 1.000)
    7       y_max      1.0457    (0.971, 1.000, 0.893)    (0.130, 0.387, 1.000)
    8       y_max      1.1166    (0.778, 1.000, 0.195)    (0.476, 0.288, 1.000)

---
## 4. Interactive 3D Path Visualization

The cell below uses `plot_paths3d_interactive` (Plotly-backed) to render the 10 acquisition paths inside the unit cube.  Each label records which boundary face the source belongs to.

If Plotly is not installed, the cell prints an informative message instead of raising an error.  Install the optional dependency with:

```
pip install plotly
```

In [10]:
try:
    from pygeoinf3d.plot import plot_paths3d_interactive

    labels = [
        f"path {i + 1}  (src: {source_face_names[i]})"
        for i in range(N_PATHS)
    ]
    fig = plot_paths3d_interactive(
        paths,
        labels=labels,
        n_samples=120,
        show_domain_box=True,
        show_markers=True,
        show_plot=True,
        title=(
            "Acquisition geometry — 10 straight rays\n"
            "(sources on all faces except z_max, receivers on z_max)"
        ),
    )

except ImportError:
    print(
        "Interactive path visualization requires Plotly, which is not installed.\n"
        "Install it with:  pip install plotly\n"
        "Then re-run this cell."
    )